# Baseline CNN for Surface Defect Detection
This notebook implements **Experimental Setting 1 (The Baseline)** from `agent.md`.
We build a simple, custom Convolutional Neural Network (CNN) from scratch to classify industrial surface defects.

**Dataset**: Severstal Steel Defect Detection (Kaggle)
**Task**: Multi-label classification — given an image, predict which of 4 defect classes are present.
**Splits**: Persisted 80/10/10 train/val/test CSVs loaded from `data/`. Split is generated once and reused.

In [ ]:
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 1 — Ensure Data Splits Exist
The split CSVs (`data/train_split.csv`, `data/val_split.csv`, `data/test_split.csv`) are generated
**once** by `split_data.py` and reused on every subsequent run. The Kaggle `test_images/` folder is
intentionally excluded because it has no ground-truth labels.

In [ ]:
DATA_ROOT   = os.path.join(os.getcwd(), 'data')
IMG_DIR     = os.path.join(DATA_ROOT, 'severstal-steel-defect-detection', 'train_images')
TRAIN_CSV   = os.path.join(DATA_ROOT, 'train_split.csv')
VAL_CSV     = os.path.join(DATA_ROOT, 'val_split.csv')
TEST_CSV    = os.path.join(DATA_ROOT, 'test_split.csv')

# Generate splits only if they do not already exist
if not all(os.path.exists(p) for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]):
    print('Split CSVs not found — running split_data.py ...')
    result = subprocess.run([sys.executable, 'split_data.py'], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('split_data.py failed. Check the output above.')
else:
    print('Split CSVs already exist — skipping split step.')

# Verify image directory
if not os.path.isdir(IMG_DIR):
    raise FileNotFoundError(
        f"Train images directory not found: {IMG_DIR}\n"
        "Make sure the dataset zip has been extracted to data/severstal-steel-defect-detection/"
    )

print(f"Train images directory: {IMG_DIR}")

## Step 2 — Configuration

In [ ]:
BATCH_SIZE    = 32
LEARNING_RATE = 1e-3
EPOCHS        = 10
IMG_SIZE      = 224
NUM_CLASSES   = 4   # Severstal has 4 defect class IDs: 1, 2, 3, 4

# Standard ImageNet normalisation (backbone-compatible)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Step 3 — Dataset
`SteelDefectDataset` reads the persisted split CSV and loads the corresponding image from `train_images/`.
Labels are **multi-hot** vectors of length 4 (one bit per defect class). An image with no annotation
in `train.csv` is treated as a **no-defect** sample (all zeros).

In [ ]:
class SteelDefectDataset(Dataset):
    """Custom Dataset for the Severstal Steel Defect Detection competition.

    Args:
        csv_path  : Path to pre-generated split CSV (train / val / test).
        img_dir   : Path to folder containing train_images.
        transform : torchvision transforms pipeline.
    """

    def __init__(self, csv_path: str, img_dir: str, transform=None):
        df = pd.read_csv(csv_path)

        # Normalise image-id column
        if 'ImageId' in df.columns:
            id_col = 'ImageId'
        elif 'ImageId_ClassId' in df.columns:
            df['ImageId'] = df['ImageId_ClassId'].apply(lambda x: x.split('_')[0])
            df['ClassId'] = df['ImageId_ClassId'].apply(lambda x: int(x.split('_')[1]))
            id_col = 'ImageId'
        else:
            raise ValueError(f'Unexpected columns in {csv_path}: {list(df.columns)}')

        # Keep only rows that actually have a mask annotation
        df = df.dropna(subset=['EncodedPixels'])

        # Build a mapping: image_name -> multi-hot label vector
        self.label_map: dict[str, np.ndarray] = {}
        for img_name, grp in df.groupby(id_col):
            label = np.zeros(NUM_CLASSES, dtype=np.float32)
            for cls_id in grp['ClassId'].astype(int):
                if 1 <= cls_id <= 4:
                    label[cls_id - 1] = 1.0   # cls_id 1-4 → index 0-3
            self.label_map[img_name] = label

        # All unique image names in this split (including no-defect images)
        all_imgs = pd.read_csv(csv_path)[id_col if id_col == 'ImageId' else 'ImageId_ClassId']
        if id_col == 'ImageId_ClassId':
            all_imgs = all_imgs.apply(lambda x: x.split('_')[0])
        self.image_names = list(all_imgs.unique())

        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_names)

    def __getitem__(self, idx: int):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Retrieve multi-hot label (default = all-zeros if not in label_map)
        label = self.label_map.get(img_name, np.zeros(NUM_CLASSES, dtype=np.float32))
        return image, torch.tensor(label, dtype=torch.float32)


# --- Build datasets and loaders ---
train_dataset = SteelDefectDataset(TRAIN_CSV, IMG_DIR, transform=train_transform)
val_dataset   = SteelDefectDataset(VAL_CSV,   IMG_DIR, transform=eval_transform)
test_dataset  = SteelDefectDataset(TEST_CSV,  IMG_DIR, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} images | Val: {len(val_dataset)} images | Test: {len(test_dataset)} images")

## Step 4 — Network Architecture
A lightweight custom CNN with 3 conv blocks, suitable for a first-pass feasibility study.

In [ ]:
class BaselineCNN(nn.Module):
    """Simple CNN from scratch for multi-label defect classification."""

    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),  # 112×112
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2), # 56×56
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2), # 28×28
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
            # NOTE: No Sigmoid here — BCEWithLogitsLoss handles it for numerical stability.
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model     = BaselineCNN(num_classes=NUM_CLASSES).to(device)
criterion = nn.BCEWithLogitsLoss()   # multi-label binary cross-entropy
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

## Step 5 — Training Loop

In [ ]:
THRESHOLD = 0.5  # Probability threshold for converting logits → binary predictions

history = {'train_loss': [], 'val_loss': [], 'val_f1': []}

for epoch in range(EPOCHS):
    # ---- Training ----
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # ---- Validation ----
    model.eval()
    val_loss  = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            val_loss += criterion(logits, labels).item()
            preds = (torch.sigmoid(logits) >= THRESHOLD).cpu().numpy().astype(int)
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy().astype(int))

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    train_loss_avg = running_loss / len(train_loader)
    val_loss_avg   = val_loss   / len(val_loader)

    history['train_loss'].append(train_loss_avg)
    history['val_loss'].append(val_loss_avg)
    history['val_f1'].append(f1)

    print(
        f"Epoch [{epoch+1:>2}/{EPOCHS}] | "
        f"Train Loss: {train_loss_avg:.4f} | "
        f"Val Loss: {val_loss_avg:.4f} | "
        f"Val F1 (macro): {f1:.4f}"
    )

## Step 6 — Test Set Evaluation & Failure Case Analysis
Reporting F1-Score, Precision, and Recall on the **held-out test split**.
The test set was never seen by the model during training or hyperparameter selection.

In [ ]:
model.eval()
test_preds  = []
test_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        preds  = (torch.sigmoid(logits) >= THRESHOLD).cpu().numpy().astype(int)
        test_preds.append(preds)
        test_labels.append(labels.numpy().astype(int))

test_preds  = np.vstack(test_preds)
test_labels = np.vstack(test_labels)

CLASS_NAMES = [f"Class {i+1}" for i in range(NUM_CLASSES)]

precision = precision_score(test_labels, test_preds, average='macro', zero_division=0)
recall    = recall_score(test_labels, test_preds, average='macro', zero_division=0)
f1_macro  = f1_score(test_labels, test_preds, average='macro', zero_division=0)

print("--- Final Test Metrics (macro-averaged) ---")
print(f"F1-Score  (macro): {f1_macro:.4f}")
print(f"Precision (macro): {precision:.4f}")
print(f"Recall    (macro): {recall:.4f}")

# Per-class breakdown
print("\n--- Per-Class Metrics ---")
for i, cls in enumerate(CLASS_NAMES):
    p = precision_score(test_labels[:, i], test_preds[:, i], zero_division=0)
    r = recall_score(test_labels[:, i], test_preds[:, i], zero_division=0)
    f = f1_score(test_labels[:, i], test_preds[:, i], zero_division=0)
    print(f"  {cls}: Precision={p:.3f}  Recall={r:.3f}  F1={f:.3f}")

In [ ]:
# --- Per-class binary confusion matrices (failure case analysis) ---
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(16, 4))
for i, ax in enumerate(axes):
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(test_labels[:, i], test_preds[:, i])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Defect', f'Class {i+1}'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Defect Class {i+1}', fontsize=12)

plt.suptitle('Confusion Matrices — Failure Case Analysis (Test Split)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n--- Discussion on Failure Cases ---")
print("Off-diagonal cells indicate misclassifications:")
print("  - Top-right: False Positives (predicted defect, but none present)")
print("  - Bottom-left: False Negatives (missed an actual defect)")
print("In a safety-critical welding environment, False Negatives are far more costly.")
print("Consider lowering the THRESHOLD to improve Recall at the expense of Precision.")

In [ ]:
# --- Training curve ---
epochs_range = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, history['train_loss'], label='Train Loss')
ax1.plot(epochs_range, history['val_loss'],   label='Val Loss')
ax1.set_title('Loss Curves')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCEWithLogitsLoss')
ax1.legend()

ax2.plot(epochs_range, history['val_f1'], color='green', label='Val F1 (macro)')
ax2.set_title('Validation F1 Score')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 (macro)')
ax2.legend()

plt.tight_layout()
plt.show()